# Entrenar M2 y M3 en Colab
Sube un ZIP de **anotaciones revisadas** por ejecución. Usa segmentación de cinco clases para M2 o pose por rueda con un centro (o el esquema anterior por vehículo) para M3. Se requiere un conjunto de prueba separado por secuencia de captura. Guarda la descarga antes de cerrar Colab.

In [ ]:
%pip install -q ultralytics==8.3.161

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, tempfile, yaml
from ultralytics import YOLO
TASK = 'pose'  # cambiar a 'segment' para M2
root=Path(tempfile.mkdtemp())
archive,=files.upload().keys()
with zipfile.ZipFile(archive) as z: z.extractall(root)
cfg_file,=root.rglob('data.yaml')
cfg=yaml.safe_load(cfg_file.read_text())
if TASK=='pose':
    assert cfg.get('kpt_shape') in ([1,3],[8,3])
    cfg['flip_idx']=list(range(cfg['kpt_shape'][0]))
else:
    names=cfg['names']
    assert set(names.values() if isinstance(names,dict) else names)=={'auto','moto','minibus','bus','camion'}
for split in ('train','val','test'):
    rel=cfg[split]
    cfg[split]=str(next(p.resolve() for p in [cfg_file.parent/rel,cfg_file.parent/rel.removeprefix('../')] if p.is_dir() and p.resolve().is_relative_to(root)))
cfg['path']=str(cfg_file.parent)
cfg_file.write_text(yaml.safe_dump(cfg))
model=YOLO('yolo11n-pose.pt' if TASK=='pose' else 'yolo11n-seg.pt')
model.train(data=str(cfg_file),epochs=100,patience=18,imgsz=640,batch=12,device=0,seed=42,mosaic=0.3,close_mosaic=8,amp=False)
best=model.trainer.best
metrics=YOLO(str(best)).val(data=str(cfg_file),split='test')
print(metrics.results_dict)
import shutil
output=shutil.copy2(best,'m3.pt' if TASK=='pose' else 'm2.pt')
files.download(output)